# Now we will enter the training phase

## 1. Dataloading

But before we can train of our preprocessed dataset, we need to load it into a trainable format ( Pytorch Dataset )

And to do that we will create a seperated script that creates a class inhereting (subclassing ) from the Torch class Dataset `torch.utils.data.Dataset class` which primarily involves implementing three key methods:

- `__init__` : initialization, its excecuted at the creation of the dataset object ( for our case we will a path variable )
- `__len__` : Total Number of Samples, used particulary for batching
- `__getitem__` : Retrieve a single sample based on the index provided (`idx`)

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
sys.path.append(str(PROJECT_ROOT))

In [4]:
import torch
from torch.utils.data import DataLoader
from scripts.dataset import PortfolioDataset,multitask_collate_fn

train_dataset = PortfolioDataset("../data/cleaned/processed/train_processed.jsonl") # this uses the __init__ method of PortfolioDataset to load the dataset from the specified JSONL file. It should read the file and prepare the data for training.

train_dataset[0] # This uses the __getitem__ method of PortfolioDataset to get the first item in the dataset. It should return a dictionary with keys like "input_ids", "attention_mask", "labels", etc., depending on how the dataset is structured.

{'input_ids': tensor([     0,  81237,  10442,   4061,  36806, 167190,  50592,    944, 106935,
             76,  59843,      5,   1529,    474,  10999,      8,  81773,  11822,
          43079,      8,    951,  63318, 120843,    253,      6, 199909,   8357,
             13,  40168,  18304,  17147,      5, 134549,   3190,    152,  29024,
          48055,      4,  35902,     56,     82,  79227,     51,  61404, 183985,
          22321,  59843,   1609,    261,  20941, 126422,    361,  33180,      7,
              5,  13538,   7870,  70770,      7,    152,     20,   1250,     67,
            223,    361,    578,     21,  40226,   1363,    115,  61404,     20,
           2027,    318,   9284,  70820,  19335,  58875,   1250,  10646,   6873,
            963,  11540,    578,     96,     25,  58345,    115,  26073,     20,
         128589,   1363,  31455,   5531,    192,    578,     96,     25, 149022,
            115, 124643, 119860,     20,  18436,    289,  15032,  44810,    613,
          48449

In [5]:
train_dataset[0].keys() # This will show the keys of the dictionary returned by __getitem__, which should give us

dict_keys(['input_ids', 'attention_mask', 'ner_labels', 'domain_labels', 'tokens', 'entities', 'original_domain_labels', 'text'])

In [6]:
train_dataset[0]["domain_labels"] # This will show the value associated with the "domain_labels" key in the first item of the dataset. It should return a tensor containing the domain labels for that item.

tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])

In [7]:
train_dataset[0]["original_domain_labels"]

['Embedded Systems and IoT']

In [8]:
len(train_dataset) # This uses the __len__ method of PortfolioDataset to get the total number of items in the dataset. It should return an integer representing the size of the dataset.

879

We will now turn our Dataset into a stream of batches that can be interable for more training efficienct (the GPU will be able to process many samples in parallel),
And also it helps shuffling our data, And with `num_workers` we allow our data to be loaded to the GPU while its training we can further optimize it allowing `pin_memory` to reduce cpu_gpu batch transfer latency.

In [9]:
train_dataloader = DataLoader(train_dataset, 
    batch_size=8, 
    shuffle=True, 
    collate_fn=multitask_collate_fn)

batch = next(iter(train_dataloader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(k, v.shape)

input_ids torch.Size([8, 126])
attention_mask torch.Size([8, 126])
ner_labels torch.Size([8, 126])
domain_labels torch.Size([8, 10])


## 2. Model loading and environnement

In [5]:
# Setup device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
import torch
import torch.nn as nn
from transformers import AutoModel
model = AutoModel.from_pretrained("xlm-roberta-base")
print(model.config)



c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


XLMRobertaConfig {
  "_name_or_path": "xlm-roberta-base",
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.35.2",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}



In [10]:
# Example to print the names of each layer block
for name, layer in model.named_children():
    print(name)

embeddings
encoder
pooler


In [11]:
# Check which parameters are trainable
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable layer: {name}")

Trainable layer: embeddings.word_embeddings.weight
Trainable layer: embeddings.position_embeddings.weight
Trainable layer: embeddings.token_type_embeddings.weight
Trainable layer: embeddings.LayerNorm.weight
Trainable layer: embeddings.LayerNorm.bias
Trainable layer: encoder.layer.0.attention.self.query.weight
Trainable layer: encoder.layer.0.attention.self.query.bias
Trainable layer: encoder.layer.0.attention.self.key.weight
Trainable layer: encoder.layer.0.attention.self.key.bias
Trainable layer: encoder.layer.0.attention.self.value.weight
Trainable layer: encoder.layer.0.attention.self.value.bias
Trainable layer: encoder.layer.0.attention.output.dense.weight
Trainable layer: encoder.layer.0.attention.output.dense.bias
Trainable layer: encoder.layer.0.attention.output.LayerNorm.weight
Trainable layer: encoder.layer.0.attention.output.LayerNorm.bias
Trainable layer: encoder.layer.0.intermediate.dense.weight
Trainable layer: encoder.layer.0.intermediate.dense.bias
Trainable layer: enco

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultitaskXLM(nn.Module):
    def __init__(self, 
                 model_name, 
                 num_domain_labels, 
                 num_ner_labels,
                 dropout = 0.1):
        super(MultitaskXLM, self).__init__()